# Model Evaluation

Evaluate all models saved by `train.py` and inspect predictions for a selected crop/version.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
from pathlib import Path


from train import (
    CROP_CONFIGS,
    ModelConfig,
    get_dataset_partitions_tf,
    latest_model_path,
    list_saved_models,
    load_dataset,
    prepare_eval_dataset,
)



def predict(model: tf.keras.Model, img: tf.Tensor, class_names: list[str]) -> tuple[str, float]:
    img_array = tf.keras.preprocessing.image.img_to_array(img.numpy())
    img_array = tf.expand_dims(img_array, 0)
    predictions = model.predict(img_array, verbose=0)
    predicted_class = class_names[predictions[0].argmax()]
    confidence = round(100 * predictions[0].max(), 2)
    return predicted_class, confidence

def prepare_eval_dataset(ds: tf.data.Dataset) -> tf.data.Dataset:
    return ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)


def list_saved_models(model_config: ModelConfig) -> list[Path]:
    if not model_config.model_dir.exists():
        return []

    return sorted(
        [
            path
            for path in model_config.model_dir.iterdir()
            if path.is_dir() and path.name.isdigit()
        ],
        key=lambda path: int(path.name),
    )


def latest_model_path(model_config: ModelConfig) -> Path:
    saved_models = list_saved_models(model_config)
    if not saved_models:
        raise FileNotFoundError(
            f"No saved models found for {model_config.name} in {model_config.model_dir}"
        )
    return saved_models[-1]


In [ ]:
results = []

for crop, model_config in CROP_CONFIGS.items():
    saved_models = list_saved_models(model_config)
    if not saved_models:
        print(f"No saved models found for {crop} at {model_config.model_dir}")
        continue

    dataset = load_dataset(model_config)
    class_names = dataset.class_names
    _, _, test_ds = get_dataset_partitions_tf(
        dataset,
        shuffle=model_config.shuffle,
        seed=model_config.seed,
    )
    test_ds = prepare_eval_dataset(test_ds)

    for model_path in saved_models:
        model = tf.keras.models.load_model(model_path)
        loss, accuracy = model.evaluate(test_ds, verbose=0)
        results.append(
            {
                "crop": crop,
                "version": int(model_path.name),
                "loss": round(float(loss), 4),
                "accuracy": round(float(accuracy), 4),
                "test_batches": len(test_ds),
                "num_classes": len(class_names),
                "model_path": str(model_path),
            }
        )

if results:
    results = sorted(results, key=lambda row: (row["crop"], row["version"]))
    try:
        import pandas as pd

        display(pd.DataFrame(results))
    except ModuleNotFoundError:
        for row in results:
            print(row)
else:
    print("No saved train.py models were found to evaluate.")


In [ ]:
SELECTED_CROP = "potato"
SELECTED_VERSION = None

selected_config = CROP_CONFIGS[SELECTED_CROP]
selected_model_path = (
    selected_config.model_dir / str(SELECTED_VERSION)
    if SELECTED_VERSION is not None
    else latest_model_path(selected_config)
)

dataset = load_dataset(selected_config)
class_names = dataset.class_names
_, _, test_ds = get_dataset_partitions_tf(
    dataset,
    shuffle=selected_config.shuffle,
    seed=selected_config.seed,
)
test_ds = prepare_eval_dataset(test_ds)

model = tf.keras.models.load_model(selected_model_path)
scores = model.evaluate(test_ds, verbose=0)

print(f"Evaluating {selected_model_path}")
print(f"Test loss: {scores[0]:.4f}")
print(f"Test accuracy: {scores[1]:.4f}")

for image_batch, label_batch in test_ds.take(1):
    first_image = image_batch[0].numpy().astype("uint8")
    first_label = label_batch[0].numpy()

    print("Actual Label:", class_names[first_label])
    plt.figure(figsize=(4, 4))
    plt.imshow(first_image)
    plt.axis("off")

    batch_predictions = model.predict(image_batch, verbose=0)
    print("\nPredicted Label:", class_names[batch_predictions[0].argmax()])
    print(batch_predictions[0])


In [ ]:
plt.figure(figsize=(15, 15))

for images, labels in test_ds.take(1):
    max_images = min(9, images.shape[0])
    for i in range(max_images):
        predicted_class, confidence = predict(model, images[i], class_names)
        actual_class = class_names[labels[i].numpy()]

        plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(f"Actual: {actual_class}\nPredicted: {predicted_class} ({confidence}%)")
        plt.axis("off")
